# Industrial Accident Insurance Medical Facility Review Exploration (Revised)

**Undergraduate prototype — revised for clarity and safe re-execution.**

Loads a public medical-facility directory, searches for a selected facility on Kakao Map, lets a person **verify the exact facility** before collecting publicly visible reviews, and optionally summarizes them using an OpenAI API. This is a demonstration, **not** a clinical assessment or a verified rating of a hospital.

**Before running:** Review Kakao's current terms and robots/access rules. Only collect content that you are permitted to access, respect rate limits, avoid collecting identifiable or sensitive content, and don't publish unverified reviews. The page selectors below are *illustrative and may need updates* as Kakao changes its UI. No real patient reviews or API credentials are embedded in this notebook.

Python packages: `pandas selenium beautifulsoup4 openai`. Install with `%pip install pandas selenium beautifulsoup4 openai` if needed. Chrome and a suitable driver/Selenium Manager must be available.

In [1]:
from pathlib import Path
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException


## 1. Load facility directory

Get the original public directory CSV from its official source and put it in the current notebook folder. The original notebook used the 2022-12-31 snapshot encoded in CP949; keep the origin and snapshot date documented. The dataset is not included here.

In [23]:
CSV_PATH = Path("근로복지공단_산재보험 지정의료기관 현황정보_20221231.csv")
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH.resolve()}")

facilities = pd.read_csv(CSV_PATH, encoding="cp949")
required_cols = {"의료기관명", "주소"}
missing = required_cols - set(facilities.columns)
if missing:
    raise ValueError(f"CSV is missing columns: {sorted(missing)}")
print(f"Loaded {len(facilities):,} facilities")
facilities[["의료기관명", "주소"]].head()


Loaded 6,059 facilities


,의료기관명,주소
0,연세대학교의과대학강남세브란스병원,서울 강남구 언주로 211 (도곡동)
1,의료법인 제일병원,서울 서초구 서초대로 116 (방배동)
2,예루살렘정형외과,서울 서초구 남부순환로350길 33 (양재동)
3,가톨릭정형외과의원,"서울 강남구 언주로 601 3층 (논현동, 파크랜드빌딩)"
4,안정형외과,서울 강남구 도산대로 303 (신사동)


## 2. Choose a facility

Use `.iloc` to select by row position. The selected name and address are *search hints*, not proof that a map result refers to the same organization.

In [24]:
HOSPITAL_INDEX = 1304  # change to another valid row position
if not 0 <= HOSPITAL_INDEX < len(facilities):
    raise IndexError("HOSPITAL_INDEX is outside the dataset")

facility = facilities.iloc[HOSPITAL_INDEX]
hospital_name = str(facility["의료기관명"]).strip()
hospital_address = str(facility["주소"]).strip()
search_query = f"{hospital_name} {' '.join(hospital_address.split()[:2])}"
print("Facility:", hospital_name)
print("Directory address:", hospital_address)
print("Search query:", search_query)


Facility: 중앙대학교 병원
Directory address: 서울 동작구 흑석로 102 (흑석동)
Search query: 중앙대학교 병원 서울 동작구


## 3. Search Kakao Map and verify the target

This opens Chrome visibly so you can compare the result name **and address** with the directory before moving to the place-details page. We do not automatically assume that the first search result is correct. Do not repeatedly automate access if the website prohibits it.

In [25]:
options = Options()
# Browser is intentionally visible for manual verification.
# No hard-coded ChromeDriver path is needed if Selenium Manager can locate a driver.
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)
try:
    driver.get("https://map.kakao.com/")
    search_box = wait.until(EC.presence_of_element_located((By.ID, "search.keyword.query")))
    search_box.clear()
    search_box.send_keys(search_query)
    search_box.send_keys(Keys.ENTER)
    print("Review the results in Chrome. Open the matching facility detail page manually.")
    print("Check both name and address before proceeding. If no exact match, stop here.")
except Exception:
    driver.quit()
    raise


Review the results in Chrome. Open the matching facility detail page manually.
Check both name and address before proceeding. If no exact match, stop here.


## 4. Read reviews currently visible in the verified detail page

After manually opening the correct facility details page, execute this cell. The selectors reflect the **original notebook's** UI and may no longer match. This extracts only reviews currently loaded in the page; it does **not** claim to retrieve all reviews or paginate automatically. If the facility detail page opens in a new tab, the code switches to it.

In [49]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    TimeoutException,
    StaleElementReferenceException
)
from bs4 import BeautifulSoup

def extract_reviews(driver):
    """Extract currently loaded visitor reviews from KakaoMap."""

    review_selector = "a.link_review p.desc_review"

    # Expand truncated reviews
    for _ in range(100):
        collapsed = [
            node for node in driver.find_elements(
                By.CSS_SELECTOR, review_selector
            )
            if "더보기" in node.text and "접기" not in node.text
        ]

        if not collapsed:
            break

        review = collapsed[0]
        link = review.find_element(By.XPATH, "./..")

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            link
        )

        try:
            link.click()

            WebDriverWait(driver, 5).until(
                lambda d: "접기" in review.text
            )

        except (TimeoutException, StaleElementReferenceException):
            print("Could not expand a review.")
            break

    # Extract review texts
    soup = BeautifulSoup(driver.page_source, "html.parser")

    reviews = []

    for node in soup.select("p.desc_review"):
        for button in node.select("span.btn_more"):
            button.decompose()

        text = node.get_text(" ", strip=True)

        if text:
            reviews.append(text)

    return reviews


reviews = extract_reviews(driver)

print(f"Collected {len(reviews)} reviews")

Collected 40 reviews


## 5. Optional summarization

Set `OPENAI_API_KEY` in your local environment **outside the notebook**. The model name is configurable because access varies by account. Never paste secrets into notebook cells or commit them to GitHub. Reviews can contain sensitive information, so review and remove identifiable details before sending content to an external API; use fictional/sample reviews for public demonstrations. The prompt treats the reviews as **unverified opinions** and prevents them from becoming asserted medical facts.

In [50]:
from openai import OpenAI

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

def summarize_reviews(review_texts, facility_name, model=MODEL):
    cleaned = [str(x).strip() for x in review_texts if str(x).strip()]
    if not cleaned:
        return "No reviews available to summarize."
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY locally before calling the API")
    client = OpenAI()  # key read from environment
    numbered = "\n".join(f"{i+1}. {r}" for i, r in enumerate(cleaned))
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": (
                "You summarize third-party reviews in Korean. Reviews are unverified personal claims, not verified facts. "
                "Attribute claims to reviewers, avoid diagnosis or asserting medical negligence as established fact, "
                "do not include personally identifying information, and do not invent details. "
                "If there are too few reviews, acknowledge the limited evidence."
            )},
            {"role": "user", "content": (
                f"Facility: {facility_name}\nSummarize only the user-provided reviews in approximately 250 Korean characters. "
                f"Use neutral formal Korean and mention both positive and negative themes when present.\n{numbered}"
            )},
        ],
        temperature=0,
    )
    return response.choices[0].message.content

# Public example uses fictional input, not a real facility review.
example_reviews = [
    "가상의 후기: 직원이 친절했다고 느꼈습니다.",
    "가상의 후기: 대기 시간이 길었다고 느꼈습니다.",
]
# Uncomment only after you have configured your key and reviewed inputs for privacy.
# example_summary = summarize_reviews(example_reviews, "가상의 의료기관")
# print(example_summary)


## 6. Finish

Close the browser explicitly. Do not save real review text or generated accusations in notebook output when publishing this repository.

In [51]:
driver.quit()
print("Browser closed.")


Browser closed.


## Limitations and provenance

- Originally an exploratory notebook using a public 2022-12-31 facility-directory snapshot, Kakao Map's browser UI, and a language-model API.
- The original notebook's hard-coded driver path, automatic first-result selection, brittle direct indexing, and embedded outputs have been revised.
- Selectors, page behavior, API availability, and website permissions can change. **This version has not been tested against the live Kakao Map UI or an OpenAI API in this environment.**
- Search ranking is not identity verification; collect only from a manually verified listing.
- Visible reviews are not a complete or representative sample; generated summaries are not medical recommendations or verified facts.
- No real review contents, sensitive information, or API keys should be committed to a public repository.
